# Module 1 — Planting Window (maize, GHA)
Estimates the **planting dekad** per maize pixel: cue-fusion green-up (Sentinel-2 NDRE + S1 SAR + FPAR) for the main seasons, or CHIRPS rainfall onset (25/20 mm) for the short rains, then the inception-report **5+7 false-start gate**.

**Where to start:** put the `planting_pipeline` folder on your Google Drive, run the cells top-to-bottom, and approve the Drive-mount and Earth-Engine sign-in prompts.

## Setup

### Stage 0 · Runtime

**What runs.** Installs the Earth Engine Python client and `geemap` into the Colab runtime. Nothing is
computed here.

**Expected output.** One line, `installed.`, after 30 to 60 s on a cold runtime. Pip warnings about
dependency resolution are normal and can be ignored.

**If it fails.** Re-run the cell. A repeated failure usually means the runtime lost its network
connection; use *Runtime → Restart session* and start again.

In [ ]:
!pip -q install earthengine-api geemap pandas geopandas 2>/dev/null
print('installed.')

### Stage 0b · Earth Engine sign-in

**What runs.** Connects to Earth Engine under the cloud project `PROJECT`. On a fresh runtime a
browser prompt appears; approve it with the Google account that has Earth Engine access.

**Expected output.** `EE ready: ok` within a few seconds. Anything else means the sign-in did not
complete.

**Which project to use.** Compute is identical across projects, but the **export queue is per
project**. `ee-manzikye` has stalled with tasks sitting in READY for hours. If you are going to
export, set `PROJECT = "indigo-proxy-484220-q8"` before running.

In [ ]:
import ee
PROJECT="ee-manzikye"
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

### Stage 0c · Pipeline code on Drive

**What runs.** Mounts Google Drive and puts `/content/drive/MyDrive/planting_pipeline` on the Python
path, so `from src import ...` resolves to the pipeline modules rather than to anything installed by
pip.

**Expected output.** `Mounted at /content/drive` followed by
`pipeline on path: /content/drive/MyDrive/planting_pipeline`.

**If you get an `AssertionError`.** The folder is not where the cell expects it. Either upload the
whole `planting_pipeline` folder to the top level of My Drive, or edit `PIPE_DIR` to the real path.
The folder must contain `run.py`, `src/` and `config/`.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys, os
PIPE_DIR="/content/drive/MyDrive/planting_pipeline"   # adjust if needed
assert os.path.isdir(PIPE_DIR), f"Upload planting_pipeline to Drive; not at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR); os.chdir(PIPE_DIR)
print("pipeline on path:", PIPE_DIR)

### Stage 0d · Run configuration

**What you choose here.**

| Variable | Meaning |
|---|---|
| `COUNTRY`, `SEASON` | select a row of `config/season_calendar.csv`; this fixes the season window and the crop calendar |
| `YEAR` | the season's planting year. A season that crosses new year (short rains, Deyr) is still keyed by its planting year |
| `S1_ORBIT` | Sentinel-1 orbit. `ASCENDING` over Kenya, because Sentinel-1B failed in 2022 and descending coverage is sparse |
| `aoi` | the whole country, from the GAUL level-0 boundary |
| `aoi_run` | the area actually computed. It ships as a **test box**, 34.4 to 37.8 E and 1.2 S to 1.2 N, about 380 by 265 km over western and central Kenya |

**Time is counted in dekads, not dates.** A dekad is a third of a month, numbered 1 to 36 through the
year: dekad 1 is 1 to 10 January, dekad 9 is 21 to 31 March, dekad 36 is 21 to 31 December. Days 21 to
the month end are one dekad, so a dekad is 8, 9, 10 or 11 days long. `utils.dekad_label(9)` prints
`9·Mar`. Where a season crosses the new year the code uses a **global dekad** `gd` running 1 to 72,
which is the dekad of `YEAR` for 1 to 36 and of `YEAR + 1` for 37 to 72.

**Season windows this notebook can use.**

| Country · season | SOS detection window | Dekads |
|---|---|---|
| Kenya · Long rains | Mar-d3 to May-d3 | 9 to 15 |
| Kenya · Short rains | Oct-d1 to Nov-d3 | 28 to 33 |
| Ethiopia · Meher | Apr-d2 to Jun-d3 | 11 to 18 |

**Expected output.** One line, for example `Kenya · Long rains · 2024 · S1 ASCENDING`.

**Before you switch to the whole country.** Replace `aoi_run` with `aoi` only when the test box has
run cleanly. The country is roughly ten times the area, and Sentinel-2 and Sentinel-1 compositing
scales with it. Expect minutes to become tens of minutes, and expect `getInfo()` calls to time out;
at country scale use `ee.batch.Export` instead of reading results back into the notebook.

In [ ]:
# --- config + GEE-native map (geemap: built-in EE Layers panel, toggle + opacity) ---
COUNTRY="Kenya"      # "Kenya" | "Ethiopia"
SEASON ="Long rains" # "Long rains" | "Short rains" | "Meher"
YEAR=2024
S1_ORBIT="ASCENDING"   # S1B gone (2022) -> ASCENDING has coverage over Kenya
from run import GAUL_NAME
from src import zonal_aggregate as ZA
aoi = ZA.gaul_admin(ee, [GAUL_NAME[COUNTRY]], level=0).geometry()
aoi_run = ee.Geometry.Rectangle([34.4,-1.2,37.8,1.2])   # fast test box; use `aoi` for whole country
import geemap
try:
    from google.colab import output; output.enable_custom_widget_manager()  # needed for interactive geemap in Colab
except Exception:
    pass
def new_map(zoom=7):
    m = geemap.Map(add_google_map=False, basemap="SATELLITE")  # keyless Google tiles; native EE layer control
    m.centerObject(aoi_run, zoom)
    return m
def ee_layer(m, image, vis, name, shown=True, opacity=1.0):
    m.addLayer(ee.Image(image), vis, name, shown, opacity)  # appears in the Layers panel (toggle + opacity)
    return m
print(f"{COUNTRY} · {SEASON} · {YEAR} · S1 {S1_ORBIT}")

## Planting-window estimation

### Stage 1 · Planting dekad

**What this stage does.** It estimates, for every maize pixel, the dekad the crop was planted. Every
later module is anchored on this number, so an error here propagates into the water balance, the CPI
and the yield. Two different methods run, chosen by season.

**Main seasons: cue-fusion green-up.** Optical greenness is combined with radar so that cloud does not
leave holes. For each dekad a fused greenness proxy is built,

$$G_t=\tfrac{1}{2}\Big[\mathrm{unit}(\mathrm{NDRE}_t;0,0.7)+\mathrm{unit}(\mathrm{FPAR}_t;0,0.9)\Big],
\qquad G_t \leftarrow \mathrm{unit}(\mathrm{RVI}_t;0.1,0.8)\ \text{where optical is missing,}$$

where $\mathrm{unit}(x;a,b)$ rescales $x$ from $[a,b]$ to $[0,1]$. NDRE is the Sentinel-2 red-edge
index, FPAR is MODIS MCD15A3H, and RVI is the Sentinel-1 radar vegetation index, which rises with
canopy and is unaffected by cloud.

Start of season is the first dekad in the window at which greenness crosses a quarter of the season's
own amplitude and is still rising:

$$G_{\text{thr}}=G_{\min}+0.25\,(G_{\max}-G_{\min}),\qquad
\mathrm{SOS}=\min\{\,t:\ G_t\ge G_{\text{thr}}\ \wedge\ G_{t+1}-G_t\ge 0\ \wedge\ |t-\mathrm{SOS}_{\mathrm{LTN}}|\le 2\,\}$$

The last condition keeps the answer within two dekads of the climatological onset, which rejects weed
flushes and a second green-up. It is applied only where a climatology exists, so a sparse second-season
normal cannot reject every pixel.

Planting precedes visible green-up, so the detected SOS is shifted back by the crop's emergence lag:

$$\text{planting dekad} = \mathrm{SOS} - 2 \quad \text{(maize; wheat and teff use 1).}$$

**Short rains: rainfall onset.** Green-up detection is unreliable in the short rains, so the FEWS NET
rule is used instead. Onset is the first dekad with

$$P_t \ge 25\ \mathrm{mm}\quad\text{and}\quad P_{t+1}+P_{t+2}\ge 20\ \mathrm{mm}
\quad\text{and}\quad P_t/ET_{0,t}\ge 0.5 .$$

The first two conditions are the classic 25/20 mm rule; the third is an agroclimatic gate that asks
whether the rain was large relative to evaporative demand.

**Expected output.** A single line, `planting dekad computed for <country> <season>`. Nothing is
evaluated yet: Earth Engine is lazy, so errors in this cell often only surface at the next one, where
a number is actually requested.

**Expected values.** The result must fall inside the SOS window of the table above, minus the
emergence offset. For Kenya long rains 2024 the modal planting dekad is **8** (11 to 20 March), with
the 10th to 90th percentile of the 253 constituencies spanning dekads **7 to 9**. A modal dekad
outside 6 to 11 for that season means the fusion locked onto the wrong green-up.

In [ ]:
# --- planting dekad (onset) — cue-fusion green-up (main seasons) or rainfall onset (short rains) ---
from src import (utils, s2_preprocess as S2, s1_preprocess as S1, fusion_phenometrics as FZ,
                 ltn as LTN, planting_date as PD, wrsi_feedback as WR)
from run import crop_mask_image
kc, soil = utils.load_crop_coeffs()
rows={(r['country'],r['season']):r for r in utils.viable_products(utils.load_calendar('config/season_calendar.csv')) if r['crop'].lower()=='maize'}
r=rows[(COUNTRY,SEASON)]; ss,se=utils.sos_window_dekads(r['sos_detection_window']); mask=crop_mask_image(ee,COUNTRY,'maize',None)
if SEASON=='Short rains':
    pet=WR.pet_dekadal(ee,aoi_run,YEAR); ch=WR.chirps_dekadal(ee,aoi_run,YEAR)
    planting=WR.wrsi_onset(ee,ch,ss,se,pet_ic=pet).updateMask(mask).toInt16()
else:
    s2=S2.build_s2_dekadal(ee,aoi_run,YEAR); s1=S1.build_s1_dekadal(ee,aoi_run,YEAR,orbit=S1_ORBIT); fpar=FZ.add_fpar_dekadal(ee,aoi_run,YEAR)
    g=FZ.build_fused_greenness(ee,s2,s1,fpar); ltn=LTN.build_ltn_prior(ee,aoi_run,ss,se)
    sos=FZ.detect_sos(ee,g,mask,ss,se,ltn_sos=ltn,ltn_pad=2); planting=PD.sos_to_planting(ee,sos,'maize').toInt16()
print('planting dekad computed for', COUNTRY, SEASON)

### Stage 2 · False-start gate, and the first real number

**What this stage does.** A first rain can germinate a crop and then stop, killing it. The inception
report's **5 + 7 rule** rejects those pixels. Both halves must hold at the estimated planting date:

* **germination trigger**: at least **20 mm** of rain in the first **5 days**;
* **continuity**: no dry spell longer than **7 days** inside the following **20 days**, where a dry day
  is one with less than 1 mm.

Pixels that fail are masked out, which is why the count printed here is lower than the number of maize
pixels in the box. The short rains do not need the gate, because the 25/20 mm onset rule already
contains a continuity test.

**Expected output.** `valid maize pixels: N`. This is the first cell that actually forces Earth Engine
to compute, so it takes the longest, typically one to three minutes for the test box.

**How to read N.** One 250 m pixel is **6.25 ha**, so `N × 6.25` is the maize area the estimate covers.
Compare it with the maize area of the box: for the shipped western-Kenya box, a result within a factor
of two of a few hundred thousand hectares is sensible.

* `N = 0` almost always means the crop mask and `aoi_run` do not overlap, or the season window is wrong
  for the country you selected.
* An N that collapses when you add the gate, rather than falling by a fraction, means the planting
  estimate is too early: the gate is then testing rain before the season began.

In [ ]:
# 5+7 false-start gate (green-up seasons; short rains already carries the 25/20 mm rule)
if SEASON!='Short rains':
    ok=WR.dryspell_false_start(ee,aoi_run,planting,YEAR,dk_lo=ss,dk_hi=se+2); planting=planting.updateMask(ok)
print('valid maize pixels:', planting.reduceRegion(ee.Reducer.count(),aoi_run,250,maxPixels=int(1e13)).get('planting_dekad').getInfo())

### Stage 3 · Map

**What this stage does.** Draws the planting dekad on an interactive map. The colour ramp runs from the
first dekad of the SOS window to three dekads past its end, so dark blue is early planting and yellow
is late. The layer panel at the top right toggles layers and sets opacity.

**What to look for.** Planting should vary smoothly with elevation and rainfall, and should be later as
you move into drier country. Salt-and-pepper noise over a small area is usually cloud; a hard straight
edge is a tile or orbit boundary and means the Sentinel-1 orbit setting is wrong for that area.

In [ ]:
M=new_map()
ee_layer(M, planting.clip(aoi_run), {'min':ss,'max':se+3,'palette':['08306b','08519c','2171b5','4292c6','6baed6','9ecae1','c6dbef']}, f'Planting dekad — {SEASON}')
M   # geemap renders its own GEE-native Layers panel (toggle + opacity slider) — no extra layer control needed

## Planting-window statistics
Distribution of the estimated planting dekad over maize area, an agreement (**skill**) score against the FEWS/FAO calendar window, and a per-admin table + ranked bar.

### Stage 4 · Planting-window statistics

**What this stage does.** Turns the map into numbers, in three parts.

**1. Distribution over maize area.** A frequency histogram of the planting dekad is converted to area,

$$A_d = n_d \times 6.25\ \mathrm{ha},$$

and summarised by the modal dekad, the area-weighted mean, and area-weighted quantiles $q$ defined as
the dekad at which the cumulative planted area first reaches $q$ of the total.

**2. Calendar agreement (skill).** The estimate is compared with the FEWS and FAO indicative planting
window for the country and season. The hit rate is the share of maize area whose estimated dekad falls
inside that window; the bias is the mean signed difference in dekads, positive meaning later than the
calendar.

**3. Per-admin table and ranked bar.** The same statistics per administrative unit, ranked, so the late
and early districts are visible.

**Expected values, Kenya long rains 2024.** Against the shipped 253-constituency table: modal dekad
**8**, area-weighted mean **8.05**, p10 **7.3**, p90 **8.8**, mean calendar hit rate **0.73**, mean
bias **−1.95 dekads** relative to the calendar window. The negative bias is expected and is a property
of the calendar, not an error: the indicative window starts at Mar-d2 while most farmers in the west
plant in Mar-d1 to Mar-d3.

**Red flags.** A hit rate below 0.3, a modal dekad at either edge of the window, or a distribution with
two widely separated peaks over a small area. The last one usually means two season regimes have been
mixed; see `KENYA_SEASON_REGIMES.md`.

In [ ]:
# ============================================================
#  Planting-window STATISTICS  (run after the map cell)
#  distribution graph · calendar-agreement skill · per-admin table & ranked bar
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
PIXEL_HA = 6.25                                   # area of one 250 m pixel
lab = utils.dekad_label                           # e.g. 9 -> "9\xb7Mar"

# ---- 1. AOI-wide planting-dekad distribution (area per dekad) --------------
hist = (planting.reduceRegion(ee.Reducer.frequencyHistogram(), aoi_run, 250,
        maxPixels=int(1e13)).get('planting_dekad').getInfo() or {})
H   = {int(round(float(k))): v for k, v in hist.items()}
dks = list(range(min(H) if H else ss, (max(H) if H else se+3) + 1))
cnt = np.array([H.get(d, 0) for d in dks], float)
area = cnt * PIXEL_HA
tot  = cnt.sum()
def wq(q):                                        # area-weighted quantile dekad
    c = np.cumsum(cnt); return int(np.array(dks)[np.searchsorted(c, tot*q)]) if tot else None
mode_dk = int(dks[int(np.argmax(cnt))]) if tot else None
mean_dk = float((cnt*np.array(dks)).sum()/tot) if tot else float('nan')

print(f"── {COUNTRY} \xb7 {SEASON} {YEAR} \xb7 planting-window statistics ──")
print(f"  maize area planted : {area.sum():,.0f} ha  ({int(tot):,} pixels)")
if tot:
    print(f"  modal dekad        : {lab(mode_dk)}")
    print(f"  median (p50) / mean: {lab(wq(0.5))}  /  {mean_dk:.1f}")
    print(f"  central 80% window : {lab(wq(0.1))}  →  {lab(wq(0.9))}   (spread {wq(0.9)-wq(0.1)} dekads)")
    inwin = area[[ss <= d <= se for d in dks]].sum()
    print(f"  SKILL — within FEWS/FAO calendar [{lab(ss)}–{lab(se)}]: {inwin/area.sum()*100:.0f}% of area")

# ---- 2. distribution bar chart (green = in calendar window, amber = outside)
fig, ax = plt.subplots(figsize=(8.4, 3.4))
ax.bar([lab(d) for d in dks], area/1000,
       color=['#3b7a57' if ss <= d <= se else '#c9a227' for d in dks])
ax.set_ylabel('maize area (000 ha)'); ax.set_xlabel('planting dekad')
ax.set_title(f'Planting-window distribution — {COUNTRY} {SEASON} {YEAR}')
ax.tick_params(axis='x', rotation=45)
for t in ax.get_xticklabels(): t.set_ha('right')
plt.tight_layout(); plt.show()

# ---- 3. per-admin planting window (GAUL level-1) --------------------------
admin = ZA.gaul_admin(ee, [GAUL_NAME[COUNTRY]], level=1).filterBounds(aoi_run)
feats = ZA.zonal_planting_stats(ee, planting, admin, scale=250).getInfo()['features']
def _key(props, suffix):                          # reduceRegions may or may not prefix with the band name 'pd'
    return next((k for k in props if k == suffix or k.endswith('_'+suffix)), None)
rows = []
if feats:
    pk = feats[0]['properties']
    kC, kMo, k10, k50, k90 = (_key(pk,'count'), _key(pk,'mode'), _key(pk,'p10'), _key(pk,'p50'), _key(pk,'p90'))
    for f in feats:
        p = f['properties']; n = (p.get(kC) or 0) if kC else 0
        if n < 20: continue                       # drop near-empty units
        gg = lambda k: lab(p[k]) if (k and p.get(k) is not None) else '—'
        rows.append(dict(Admin=p.get('ADM1_NAME','?'), _med=(p.get(k50) if k50 else None),
                         Modal=gg(kMo), Median=gg(k50), Early_p10=gg(k10), Late_p90=gg(k90),
                         Area_ha=round(n*PIXEL_HA)))
if not rows:
    print('  (no admin unit has >=20 maize pixels in aoi_run — set aoi_run = aoi in the config cell for the whole country)')
else:
    df = pd.DataFrame(rows).sort_values('_med', na_position='last').reset_index(drop=True)
    display(df.drop(columns='_med'))
    d2 = df.dropna(subset=['_med'])
    if len(d2):
        fig, ax = plt.subplots(figsize=(7.5, max(2.4, 0.34*len(d2))))
        ax.barh(d2['Admin'], d2['_med'], color='#3b528b'); ax.invert_yaxis()
        ax.set_xlabel('median planting dekad')
        for y, m in zip(range(len(d2)), d2['_med']):
            ax.text(m, y, ' '+lab(int(m)), va='center', fontsize=8)
        ax.set_title(f'Median planting dekad by admin — {COUNTRY} {SEASON}')
        plt.tight_layout(); plt.show()

## Farmer validation — 2024 MAM
Compares the estimated planting dekad with **farmer-reported** planting per Kenyan county (bias, MAE, ±2-dekad hit-rate). Runs only for **Kenya · Long rains**; uses your live estimate where `aoi_run` covers enough counties, otherwise the shipped validation CSV.

### Stage 5 · Farmer validation, 2024 long rains

**What this stage does.** Compares the estimate with what farmers reported, which is the only
independent test in this notebook. It runs for Kenya long rains only. Three metrics, per county,
weighted by the number of farmers surveyed:

$$\text{bias}=\overline{\hat{d}-d},\qquad
\mathrm{MAE}=\overline{|\hat{d}-d|},\qquad
\text{hit rate}=\Pr\big(|\hat{d}-d|\le 2\ \text{dekads}\big),$$

with $\hat d$ the estimated and $d$ the farmer-reported planting dekad. A tolerance of two dekads is
used because farmers report a planting month or a modal date, not a day.

**Expected values.** The shipped county file has **42 counties**: bias **−0.31 dekads**, MAE **1.02
dekads**, and **93 %** of counties within two dekads. In other words the estimate is on average three
days early and typically wrong by about ten days, which is inside the precision at which farmers
report.

**Interpretation.** An MAE near 1 dekad is the working accuracy of this product. Do not read a
one-dekad difference between two counties as real. Do read a three-dekad difference as real.

**If the live check is skipped.** Where `aoi_run` does not cover enough counties, the cell falls back
to the shipped CSV and says so. That is the documented number above, not a check on the run you just
did.

In [ ]:
# ============================================================
#  Farmer validation — 2024 MAM (Kenya long rains) planting dates
#  estimated vs observed farmer planting per county; bias / MAE / hit-rate
#  prefers a LIVE check of the estimate you just computed; falls back to the shipped CSV
# ============================================================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
VAL_CSV = 'planting_validation_MAM_2024.csv'      # ships in the pipeline folder (on Drive)
if not (COUNTRY == 'Kenya' and 'ong' in SEASON):
    print('Farmer validation is for Kenya \xb7 Long rains (2024 MAM) only — skipping for', COUNTRY, SEASON)
elif not os.path.exists(VAL_CSV):
    print('Validation CSV not found at', os.path.abspath(VAL_CSV), '— upload it with the pipeline folder.')
else:
    obs = pd.read_csv(VAL_CSV)
    obs['County'] = obs['County'].astype(str).str.upper().str.strip()

    # LIVE: median estimated dekad per county from the planting image you just computed
    def _key(props, suffix): return next((k for k in props if k == suffix or k.endswith('_'+suffix)), None)
    adm = ZA.gaul_admin(ee, ['Kenya'], level=1).filterBounds(aoi_run)
    fe  = ZA.zonal_planting_stats(ee, planting, adm, scale=250).getInfo()['features']
    est = []
    if fe:
        kC, k50 = _key(fe[0]['properties'],'count'), _key(fe[0]['properties'],'p50')
        for f in fe:
            p = f['properties']; n = (p.get(kC) or 0) if kC else 0
            if n >= 20 and k50 and p.get(k50) is not None:
                est.append((str(p.get('ADM1_NAME','')).upper().strip(), float(p[k50])))
    est = pd.DataFrame(est, columns=['County','est_dk'])
    m = obs.merge(est, on='County', how='inner')
    if len(m) >= 5:
        src = f'LIVE — this run, {len(m)} counties in aoi_run'
    else:                                          # test box: too few counties -> use the shipped estimate
        m = obs.assign(est_dk=obs['modal_dekad']); src = f'shipped CSV estimate, {len(m)} counties'
    m = m.dropna(subset=['obs_dk','est_dk'])
    err = m['est_dk'] - m['obs_dk']                # estimated − observed (dekads)
    bias, mae, within2 = err.mean(), err.abs().mean(), (err.abs() <= 2).mean()*100
    hit = m['hit_rate'].mean()*100 if 'hit_rate' in m else float('nan')
    print(f"── Kenya \xb7 MAM 2024 farmer validation ({src}) ──")
    print(f"  bias (est−obs) : {bias:+.2f} dekads    MAE : {mae:.2f} dekads")
    print(f"  counties within \xb12 dekads : {within2:.0f}%    mean pixel hit-rate : {hit:.0f}%")

    # scatter: observed vs estimated, 1:1 line + ±2 dekad band
    lo = int(min(m['obs_dk'].min(), m['est_dk'].min())) - 1
    hi = int(max(m['obs_dk'].max(), m['est_dk'].max())) + 1
    fig, ax = plt.subplots(figsize=(4.7, 4.7))
    ax.fill_between([lo,hi], [lo-2,hi-2], [lo+2,hi+2], color='#3b7a57', alpha=0.12, label='\xb12 dekads')
    ax.plot([lo,hi], [lo,hi], 'k--', lw=1, label='1:1')
    ax.scatter(m['obs_dk'], m['est_dk'], s=30, color='#a50026', edgecolor='w', zorder=3)
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi); ax.set_aspect('equal')
    ax.set_xlabel('observed farmer planting (dekad)'); ax.set_ylabel('estimated planting (dekad)')
    ax.set_title('MAM 2024 · estimate vs farmers'); ax.legend(fontsize=8, loc='upper left')
    for _, r in m.iterrows():
        ax.annotate(r['County'].title(), (r['obs_dk'], r['est_dk']), fontsize=6, alpha=0.6,
                    xytext=(2,2), textcoords='offset points')
    plt.tight_layout(); plt.show()

    # worst-agreement counties
    worst = m.assign(err=err).reindex(err.abs().sort_values(ascending=False).index).head(8)
    cols = [c for c in ['County','obs_dk','est_dk','err','hit_rate'] if c in worst.columns]
    tbl = worst[cols].copy()
    tbl.columns = ['County','Observed dk','Estimated dk','Error (est−obs)'] + (['Hit-rate'] if 'hit_rate' in cols else [])
    display(tbl.reset_index(drop=True))

## County → ward drill-down
Ward-level (2024 MAM) validation, a nested **county→ward JSON**, and a per-county observed-vs-estimated bar. Change `COUNTY_PICK` to drill into any county.

### Stage 6 · Ward drill-down

**What this stage does.** Repeats the validation at ward level, writes a nested county to ward JSON for
the dashboard, and draws observed against estimated planting per county.

**Expected values.** The shipped ward file holds **855 wards across 42 counties**: farmer-weighted bias
**−0.38 dekads**, MAE **1.09 dekads**, and **96 %** of wards within two dekads. Ward-level error is
almost identical to county level, which tells you the residual error is not a boundary or aggregation
artefact but the genuine spread of planting within a season.

**Using the JSON.** The structure is `{County: {Ward: {obs, est, err, farmers, n_px}}}`. Wards with a
small `n_px` carry few maize pixels, so their estimate is noisy; filter on it before ranking wards.

In [ ]:
# ============================================================
#  County → Ward drill-down + ward-level validation   (2024 MAM, Kenya)
#  ward validation metrics · nested county→ward JSON · per-county grouped bar
# ============================================================
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
WARD_CSV = 'planting_validation_MAM_ward_2024.csv'      # ships in the pipeline folder (on Drive)
if not os.path.exists(WARD_CSV):
    print('Ward CSV not found at', os.path.abspath(WARD_CSV), '— upload it with the pipeline folder.')
else:
    w = pd.read_csv(WARD_CSV).dropna(subset=['obs_dk','modal_dekad'])
    w['County'] = w['County'].astype(str).str.upper().str.strip()
    w['err'] = w['modal_dekad'] - w['obs_dk']              # estimated − observed (dekads)
    fw = w['Farmers (n)'].fillna(0).clip(lower=0)

    # ward-level validation, weighted by number of farmers surveyed
    bias = (w['err']*fw).sum()/max(fw.sum(),1); mae = (w['err'].abs()*fw).sum()/max(fw.sum(),1)
    within2 = (w['err'].abs() <= 2).mean()*100
    print(f"── Ward-level farmer validation · {len(w)} wards · {w['County'].nunique()} counties ──")
    print(f"  farmer-weighted bias {bias:+.2f} dk · MAE {mae:.2f} dk · wards within \xb12 dekads: {within2:.0f}%")

    # nested county -> ward JSON  {County: {Ward: {obs, est, err, farmers, n_px}}}
    nested = {}
    for _, r in w.iterrows():
        farmers = int(r['Farmers (n)']) if pd.notna(r['Farmers (n)']) else 0
        nested.setdefault(r['County'].title(), {})[str(r['Ward']).title()] = dict(
            obs=int(r['obs_dk']), est=int(r['modal_dekad']), err=int(r['err']),
            farmers=farmers, n_px=int(r['n_px']) if pd.notna(r.get('n_px')) else 0)
    with open('planting_ward_MAM_2024.json','w') as f: json.dump(nested, f, indent=1)
    print(f"  wrote planting_ward_MAM_2024.json ({len(nested)} counties, {len(w)} wards)")

    # ---- interactive drill-down: choose a county from the dropdown ---------
    def draw_county(county):
        sub = w[w['County'] == county].sort_values('obs_dk')
        if not len(sub):
            print('No wards for', county); return
        x = np.arange(len(sub)); bwid = 0.4
        fig, ax = plt.subplots(figsize=(max(6, 0.42*len(sub)), 4))
        ax.bar(x-bwid/2, sub['obs_dk'],      bwid, label='observed (farmers)', color='#3b7a57')
        ax.bar(x+bwid/2, sub['modal_dekad'], bwid, label='estimated',          color='#a50026')
        ax.set_xticks(x); ax.set_xticklabels(sub['Ward'].str.title(), rotation=60, ha='right', fontsize=7)
        ax.set_ylabel('planting dekad'); ax.legend(fontsize=8)
        fwc = sub['Farmers (n)'].fillna(0)
        b = (sub['err']*fwc).sum()/max(fwc.sum(),1)
        ax.set_title(f"{county.title()} — planting by ward (2024 MAM) · bias {b:+.1f} dk · {len(sub)} wards")
        plt.tight_layout(); plt.show()
        show = sub[['Ward','obs_dk','modal_dekad','err','Farmers (n)']].rename(
            columns={'obs_dk':'Observed','modal_dekad':'Estimated','err':'Error (est−obs)','Farmers (n)':'Farmers'})
        display(show.reset_index(drop=True))
    opts = [(c.title(), c) for c in sorted(w['County'].unique())]
    try:
        import ipywidgets as widgets
        try:
            from google.colab import output as _o; _o.enable_custom_widget_manager()   # Colab: enable interactive widgets
        except Exception: pass
        widgets.interact(draw_county, county=widgets.Dropdown(options=opts, value='BUNGOMA', description='County:'))
    except Exception as e:                                  # no ipywidgets -> static fallback
        print('(interactive dropdown unavailable:', e, ') — showing BUNGOMA; call draw_county("KISUMU") to change')
        draw_county('BUNGOMA')

*Higher dekad = later planting. Export with `ee.batch.Export.image.toDrive(...)`; see `run.py` for batch runs.*